## メイン関数

In [34]:
# --- ここにファイルパスを指定 ---
dat_file = "complex.dat"
solution_file = "sol_complex (1).txt"

# 1. ファイルからデータをパース
print(f"'{dat_file}' からリズム情報を読み込み中...")
bpm, rhythms = parse_dat_file(dat_file)

print(f"'{solution_file}' から運指データを読み込み中...")
fingerings = parse_solution_file(solution_file)

# 2. データが正しく読み込めた場合のみタブ譜を生成
if bpm is not None and rhythms and fingerings:
    print("\n--- 読み込み成功 ---")
    generate_rhythmic_tab(bpm, rhythms, fingerings)
else:
    print("\nエラー: ファイルの読み込みに失敗したため、タブ譜を生成できませんでした。")

'complex.dat' からリズム情報を読み込み中...
'sol_complex (1).txt' から運指データを読み込み中...

--- 読み込み成功 ---
--- TAbLATURE (BPM: 120) ---
Rhythm Key: s=16th, st=16th Triplet, e=8th, et=8th Triplet, q=Quarter
------------------------------------------------------------------------------------

Rhythm: START-e-----e-----e-----q---------q---------q---------q---------s---s---e-----
e|------------------------------------------------------------------------------
B|------------------10----------------------------------------------------------
G|------------------------------------------------------9---------10--9---------
D|12----10----12----------12--------10--------12--------------------------12----
A|------------------------------------------------------------------------------
E|------------------------------------------------------------------------------
Finger: 4-----2-----4-----2-----4---------2---------4---------1---------2---1---4-----

--------------------------------------------------------------------

## Import libraries

In [8]:
import re
import math

## .datファイルからリズムを抽出

In [9]:
def parse_dat_file(dat_filepath):
    """
    .datファイルからBPMとavailableTime (ms) のリストを抽出します。
    """
    bpm = 120 # デフォルト
    available_time_ms = []

    try:
        with open(dat_filepath, 'r') as f:
            content = f.read()

        # BPMの抽出
        bpm_match = re.search(r'// BPM: (\d+)', content)
        if bpm_match:
            bpm = int(bpm_match.group(1))

        # availableTimeの抽出
        time_match = re.search(r'availableTime = \[(.*?)\];', content, re.DOTALL)
        if time_match:
            time_str = time_match.group(1).strip()
            # 空白や改行を無視して数値のみをリスト化
            available_time_ms = [int(t.strip()) for t in time_str.split(',') if t.strip()]
        
        if not available_time_ms:
            print(f"警告: {dat_filepath} から 'availableTime' が見つかりませんでした。")

    except FileNotFoundError:
        print(f"エラー: {dat_filepath} が見つかりません。")
        return None, None
    except Exception as e:
        print(f"エラー: {dat_filepath} の読み込み中にエラーが発生しました: {e}")
        return None, None
        
    return bpm, available_time_ms

## .txtファイルから実行結果を抽出

In [32]:
def parse_solution_file(solution_filepath):
    """
    CPLEXの出力(.txt)から運指データ（String, Fret, Finger）を抽出します。
    """
    notes_to_play = []
    
    try:
        with open(solution_filepath, 'r') as f:
            lines = f.readlines() 

        header_found = False
        for line in lines:
            line = line.strip()
            if not line:
                continue
                
            # 'Note String Fret Handposition Finger' を探す
            if re.match(r'Note\s+String\s+Fret\s+Handposition\s+Finger', line):
                header_found = True
                continue 
            
            if not header_found:
                continue

            if ',' in line or '<<<' in line:
                break
                
            parts = re.split(r'\s+', line.strip())
            
            # 5列あることを確認
            if len(parts) >= 5:
                try:
                    string_num = int(parts[1])
                    fret_num = int(parts[2])
                    finger_num = int(parts[4]) # 5列目から指番号を取得
                    
                    notes_to_play.append({
                        'string': string_num, 
                        'fret': fret_num,
                        'finger': finger_num # 指番号を辞書に追加
                    })
                except (ValueError, IndexError):
                    pass
        
        if not notes_to_play:
             print(f"警告: {solution_filepath} から運指データを抽出できませんでした。ヘッダーは見つかりましたか？")

    except FileNotFoundError:
        print(f"エラー: {solution_filepath} が見つかりません。")
        return []
    except Exception as e:
        print(f"エラー: {solution_filepath} の読み込み中にエラーが発生しました: {e}")
        return []

    return notes_to_play

## 二つのデータを併せてタブ譜を作成

In [33]:
def generate_rhythmic_tab(bpm, available_time_ms, notes_to_play):
    """
    パースされたデータからテキストタブ譜を生成します。
    BPMに基づきリズムを自動計算し、指番号も表示します。
    """

    # --- 1. BPMに基づき、リズムの「基準値(ms)」を計算 ---
    quarter_note_ms = 60000.0 / bpm
    
    # リズム定義: 記号: (基準となる音価(4分音符=1.0), 視覚的幅(文字数))
    rhythm_definitions = {
        "q":  (1.0, 10),      # 4分音符
        "et": (2.0/3.0, 8),   # 8分3連符
        "e":  (0.5, 6),       # 8分音符
        "st": (1.0/3.0, 4),   # 16分3連符 (8分3連の半分)
        "s":  (0.25, 4),      # 16分音符
        "R":  (0.0, 2)        # 0ms (休符/装飾音)
    }

    # BPMから計算した「理想のms値」と「記号/幅」のリストを作成
    # (例: BPM 120なら [(500.0, ('q', 10)), (333.3, ('et', 8)), ...])
    bpm_scaled_rhythms = sorted(
        [(quarter_note_ms * info[0], (symbol, info[1])) for symbol, info in rhythm_definitions.items()],
        key=lambda x: x[0] # ms値でソート
    )

    def get_rhythm_info(ms_value):
        """入力されたms値に最も近いリズム定義を探す"""
        
        if ms_value >= 900: # 1000ms (START)
            return "START", 6

        # 最も近いms値を持つリズム定義を見つける
        closest_match = min(
            bpm_scaled_rhythms, 
            key=lambda x: abs(x[0] - ms_value)
        )
        
        ideal_ms, (symbol, width) = closest_match
        
        # 誤差が大きすぎる場合は「不明」とする (理想値の25%以内)
        if abs(ideal_ms - ms_value) > (ideal_ms * 0.25) and ideal_ms > 0:
             # dat_generator.py (111ms, 148ms) と complex.dat (125ms, 250ms) の
             # どちらのロジックにも当てはまらない場合
             print(f"不明なリズム値: {ms_value}ms (BPM {bpm} で計算外)")
             return "?", 4
        
        return symbol, width

    # --- 2. タブ譜の構築と出力 ---
    print(f"--- TAbLATURE (BPM: {bpm}) ---")
    print("Rhythm Key: s=16th, st=16th Triplet, e=8th, et=8th Triplet, q=Quarter")
    print("-" * (75+9) + "\n")

    tab_lines = {
        'rhythm': [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: [],
        'finger': [] # 指番号の行を追加
    }
    line_width = 75 
    current_line_length = 0

    num_notes = len(notes_to_play)
    if num_notes != len(available_time_ms):
        print(f"警告: 音符の数が一致しません。運指: {num_notes}, リズム: {len(available_time_ms)}")
        num_notes = min(num_notes, len(available_time_ms))

    for i in range(num_notes):
        note_info = notes_to_play[i]
        
        # 1. 情報を取得
        rhythm_symbol, padding_width = get_rhythm_info(available_time_ms[i])
        s = note_info['string']    # 弦 (1-6)
        f_str = str(note_info['fret'])   # フレット (文字列)
        h_str = str(note_info['finger']) # 指番号 (文字列)
        
        # 2. 必要な幅を決定 (リズム、フレット、指番号のうち最も幅が必要なものに合わせる)
        col_width = max(len(rhythm_symbol), len(f_str), len(h_str), padding_width - 1) + 1

        # 3. 各行にデータを追加
        tab_lines['rhythm'].append(rhythm_symbol.ljust(col_width, '-'))
        tab_lines['finger'].append(h_str.ljust(col_width, '-'))
        
        for string_index in range(1, 7):
            if string_index == s:
                tab_lines[string_index].append(f_str.ljust(col_width, '-'))
            else:
                tab_lines[string_index].append("-" * col_width)
                
        current_line_length += col_width
        
        # 4. 1行の最大幅を超えたら、出力して改行
        if current_line_length > line_width:
            print("Rhythm: " + "".join(tab_lines['rhythm']))
            print("e|" + "".join(tab_lines[1]))
            print("B|" + "".join(tab_lines[2]))
            print("G|" + "".join(tab_lines[3]))
            print("D|" + "".join(tab_lines[4]))
            print("A|" + "".join(tab_lines[5]))
            print("E|" + "".join(tab_lines[6]))
            print("Finger: " + "".join(tab_lines['finger'])) # 指番号行を出力
            print("\n" + ("-"* (line_width+2)) + "\n")
            
            tab_lines = { 'rhythm': [], 1: [], 2: [], 3: [], 4: [], 5: [], 6: [], 'finger': [] }
            current_line_length = 0

    # 5. 最後の行を出力
    if current_line_length > 0:
        print("Rhythm: " + "".join(tab_lines['rhythm']))
        print("e|" + "".join(tab_lines[1]))
        print("B|" + "".join(tab_lines[2]))
        print("G|" + "".join(tab_lines[3]))
        print("D|" + "".join(tab_lines[4]))
        print("A|" + "".join(tab_lines[5]))
        print("E|" + "".join(tab_lines[6]))
        print("Finger: " + "".join(tab_lines['finger']))